# 05 — Time Series: Income Volatility

Models monthly income cycles per worker cluster and detects early displacement signals using Facebook Prophet.

Install:
```
pip install prophet
```

In real usage, replace the synthetic income series with CMIE CPHS monthly income records per household cluster.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from src.graph.builder import load_data

workers, _, _, _, edges = load_data(use_synthetic=True)

def synthetic_income_series(base_income, volatility, n_months=36, seed=0):
    np.random.seed(seed)
    trend = np.linspace(base_income, base_income * (1 - volatility * 0.3), n_months)
    noise = np.random.normal(0, base_income * volatility * 0.5, n_months)
    seasonal = base_income * 0.08 * np.sin(np.linspace(0, 4 * np.pi, n_months))  # semi-annual cycle
    return np.clip(trend + noise + seasonal, base_income * 0.3, None)

end_date = datetime(2026, 5, 1)
dates = [end_date - timedelta(days=30 * i) for i in range(35, -1, -1)]

print('Synthetic income series generated for', len(workers), 'worker clusters')

In [ ]:
# Plot income trajectories for all clusters
fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)
axes = axes.flatten()

for i, (_, row) in enumerate(workers.iterrows()):
    series = synthetic_income_series(row['avg_monthly_income'], row['income_volatility'], seed=i)
    color = '#e74c3c' if row['risk_label'] == 1 else '#27ae60'
    axes[i].plot(dates, series, color=color, linewidth=1.2)
    axes[i].set_title(row['label'], fontsize=8)
    axes[i].set_ylabel('Income (Rs)', fontsize=7)
    axes[i].tick_params(axis='x', rotation=30, labelsize=6)
    axes[i].tick_params(axis='y', labelsize=7)

fig.suptitle('Monthly income trajectories by worker cluster (red = high-risk)', fontsize=10)
plt.tight_layout()
plt.savefig('../logs/income_trajectories.png', bbox_inches='tight')
plt.show()

In [ ]:
# Compute and compare coefficient of variation (income volatility proxy)
from src.graph.features import compute_income_volatility

results = []
for i, (_, row) in enumerate(workers.iterrows()):
    series = synthetic_income_series(row['avg_monthly_income'], row['income_volatility'], seed=i)
    cv = compute_income_volatility(series)
    results.append({'label': row['label'], 'computed_volatility': cv,
                    'stated_volatility': row['income_volatility'], 'risk': row['risk_label']})

vol_df = pd.DataFrame(results).sort_values('computed_volatility', ascending=False)
print(vol_df.to_string(index=False))

In [ ]:
# Prophet forecasting on a single high-risk cluster
try:
    from prophet import Prophet

    # Use gig delivery riders (index 0) as the example
    row = workers.iloc[0]
    series = synthetic_income_series(row['avg_monthly_income'], row['income_volatility'], seed=0)

    prophet_df = pd.DataFrame({'ds': pd.to_datetime(dates), 'y': series})
    m = Prophet(seasonality_mode='multiplicative', yearly_seasonality=True, weekly_seasonality=False)
    m.fit(prophet_df)

    future  = m.make_future_dataframe(periods=12, freq='MS')
    forecast = m.predict(future)

    fig2 = m.plot(forecast)
    fig2.axes[0].set_title(f'Income forecast — {row["label"]}')
    fig2.axes[0].set_xlabel('Date')
    fig2.axes[0].set_ylabel('Monthly income (Rs)')
    plt.tight_layout()
    plt.savefig('../logs/prophet_forecast.png', bbox_inches='tight')
    plt.show()

    print('12-month forecast tail:')
    print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12).to_string(index=False))

except ImportError:
    print('Prophet not installed. Run: pip install prophet')